Data Exploration of medici_transactions.csv/.json files:

## Data Generation and Validation Summary

Three Python scripts were used to prepare and validate the Medici Bank transaction dataset.

### 1. Initial historical data generation

The `generate_historical_data.py` script created 20,000 synthetic, historically themed banking transactions covering 1390–1440. The dataset includes deposits, withdrawals, loans, loan repayments, bills of exchange, operating expenses, alum trading, war financing, and the Council of Constance ransom payment.

### 2. Dataset expansion and embezzlement trail

The `generate_additional_data.py` script added 60,000 legitimate transactions and 230 intentionally suspicious operating-expense transactions. These suspicious payments:

- Total 95,610 florins
- Occur between January 3, 1420 and December 26, 1424
- Originate exclusively from the Florence branch
- Are paid to the fictitious vendor `Ser Benedetto Forniture`
- Use repetitive, mostly round payment amounts and occur at unusually regular intervals

After expansion, the dataset contains 80,230 transactions.

### 3. Data validation

The `validate_transactions.py` script confirmed that both the CSV and JSON datasets are structurally valid. All 80,230 transactions passed the accounting validation with no errors.

- Total debits: 21,612,302,640.94 florins
- Total credits: 21,612,302,640.94 florins
- Debit-credit difference: 0.00 florins
- CSV validation: Passed
- JSON validation: Passed
- Date coverage: 1390–1440, spanning 51 years

The equality of total debits and credits confirms that the dataset follows double-entry accounting principles.

1. **CSV - Load & Structure Check**
2. **Purpose**: Load CSV file into DF and check to see how the data is organized/structured
3. **Why**: To investigate how many records, columns, etc are present in the dataset
4. **Method**: Use basic panda exploration commands like head, tail, info to get a better idea of the dataset
5. **Findings**:
- After running generate_historical_data.py found only 20K historical  transactions then ran generate_additonal_data.py to load another 60K transactions which expanded to over 80K with a 230 embezzlement transactions.
- Total transactions after validating data = 80,230.
- There are 13 columns contain relevant information: referencing branch, counterparty, credit_account, credit_account_2, credit_amount, credit_amount_2, currency, date, debit_account, debit_amount, description, id, type.
- There 65,632 null transactions credit_account_2/credit_amount_2. Transactions cover 1390-01-01 to 1440-12-31.

In [1]:
import pandas as pd

In [2]:
df_csv = pd.read_csv("medici_transactions.csv")

In [3]:
print("Data loaded sucessfully!")

Data loaded sucessfully!


In [5]:
df_csv.shape

(80230, 13)

In [7]:
df_csv.head()

,branch,counterparty,credit_account,credit_account_2,credit_amount,credit_amount_2,currency,date,debit_account,debit_amount,description,id,type
0,Florence,Republic of Florence,Cash,NaN,82833.66,NaN,florin,1390-01-01,Loans Receivable - Government,82833.66,Emergency war financing for Florence defense,1,war_financing
1,Florence,Republic of Florence,Cash,NaN,124432.65,NaN,florin,1390-01-01,Loans Receivable - Government,124432.65,Loan to Venice for Lombardy Wars operations,2,war_financing
2,Bruges,Gold Merchant,Cash,NaN,42534.83,NaN,florin,1390-01-01,Loans Receivable,42534.83,Loan issued to Gold Merchant from Bruges branch,3,loan_issuance
3,Florence,Republic of Florence,Cash,NaN,1617678.46,NaN,florin,1390-01-01,Loans Receivable - Government,1617678.46,War financing for Florentine operations agains...,4,war_financing
4,Florence,Republic of Florence,Cash,NaN,27742.84,NaN,florin,1390-01-01,Loans Receivable - Government,27742.84,Loan to Venice for Lombardy Wars operations,5,war_financing


In [ ]:
df_csv.tail()

In [ ]:
df_csv.columns.tolist()

In [ ]:
df_csv.info()

In [ ]:
df_csv.isnull().sum()

In [11]:
df_csv["date"] = pd.to_datetime(df_csv["date"])

print("Date data type:", df_csv["date"].dtype)
print("Earliest transaction:", df_csv["date"].min())
print("Latest transaction:", df_csv["date"].max())

Date data type: datetime64[us]
Earliest transaction: 1390-01-01 00:00:00
Latest transaction: 1440-12-31 00:00:00


2. **CSV - Currency Consistency Check**
- Purpose: Check to see if there a different currency types within the dataset
- Why: To confirm sure the same currency is being used across the transactions within the dataset
- Method: Noticed currency was a column in the dataset so used value_count to see if all transactions had the same currency
- Findings: There is only 1 currency type across the whole data set, all 80,230 rows

In [12]:
df_csv["currency"].value_counts()

currency
florin    80230
Name: count, dtype: int64

3. **CSV — Branch Distribution**
- **Purpose:** Break down transaction volume by branch to understand which locations dominate the dataset.
- **Why:** Branch is a key filter needed for the dashboard and a required field on every Cleaned Transaction — need to confirm the branch values are clean and match expectations before ingestion module is created.
- **Method:** `value_counts()` on `branch`, both raw counts and normalized percentages.
- **Findings:** Rome leads at 32.77% (26,288 transactions), Florence second at 22.80% (18,295), then Venice, Bruges, London, Avignon, Milan, and Geneva clustered between 6.95%–9.23%. Constance has exactly 1 transaction (0.00%), which was confirmed against the branch and `ransom_payment`. These percentages line up closely with the expected distribution in `TRANSACTION_DATA.md` (Rome 33.1%, Florence 22.1%, Venice 9.7%), so no major deviation flagged here.

In [13]:
branch_counts = df_csv["branch"].value_counts()

branch_counts

branch
Rome         26288
Florence     18295
Venice        7407
Bruges        5700
London        5663
Avignon       5663
Milan         5638
Geneva        5575
Constance        1
Name: count, dtype: int64

In [14]:
branch_percentages = df_csv["branch"].value_counts(normalize=True).mul(100).round(2)

branch_percentages

branch
Rome         32.77
Florence     22.80
Venice        9.23
Bruges        7.10
London        7.06
Avignon       7.06
Milan         7.03
Geneva        6.95
Constance     0.00
Name: proportion, dtype: float64

In [ ]:
df_csv[df_csv["branch"] =="Constance"]

In [16]:
df_csv[df_csv["type"] =="ransom_payment"]

,branch,counterparty,credit_account,credit_account_2,credit_amount,credit_amount_2,currency,date,debit_account,debit_amount,description,id,type
39791,Constance,Council of Constance - Pope John XXIII Ransom,Cash,NaN,35000.0,NaN,florin,1415-05-29,Papal Receivable,35000.0,"Payment of 35,000 florin ransom for Pope John ...",39792,ransom_payment


## 4. CSV — Transaction Type Distribution

- **Purpose:** Break down transaction volume by type to see which categories make up the dataset.
- **Why:** `type` must be one of the 9 values defined in the data contract (`deposit`, `withdrawal`, `loan_issuance`, `loan_repayment`, `operating_expense`, `war_financing`, `bill_of_exchange`, `alum_trade`, `ransom_payment`) — confirming the value set and distribution here catches any stray/typo'd type before ingestion.
- **Method:** `value_counts()` on `type`, both raw counts and normalized percentages.
- **Findings:** All 9 expected types are present, no unexpected values. `deposit` is largest at 31.20% (25,031), followed by `operating_expense` at 13.69% (10,987) and `war_financing` at 13.17% (10,568). `ransom_payment` is a single transaction (0.00%), consistent with the one historical ransom event. Distribution closely matches `TRANSACTION_DATA.md`'s expected percentages. 

In [17]:
transaction_type_counts = df_csv["type"].value_counts()

transaction_type_counts

type
deposit              25031
operating_expense    10987
war_financing        10568
loan_repayment        8170
loan_issuance         7478
bill_of_exchange      6428
withdrawal            6374
alum_trade            5193
ransom_payment           1
Name: count, dtype: int64

In [18]:
transaction_type_percentages=(
    df_csv["type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

transaction_type_percentages

type
deposit              31.20
operating_expense    13.69
war_financing        13.17
loan_repayment       10.18
loan_issuance         9.32
bill_of_exchange      8.01
withdrawal            7.94
alum_trade            6.47
ransom_payment        0.00
Name: proportion, dtype: float64

5. **JSON Load and Structure Check:**
- Purpose:
- Why:
- Method:
- Findings: 


In [23]:
df_json = pd.read_json('medici_transactions.json')

In [24]:
df_json.head()

,branch,counterparty,credit_account,credit_account_2,credit_amount,credit_amount_2,currency,date,debit_account,debit_amount,description,id,type
0,Florence,Republic of Florence,Cash,,82833.66,,florin,1390-01-01,Loans Receivable - Government,82833.66,Emergency war financing for Florence defense,1,war_financing
1,Florence,Republic of Florence,Cash,NaN,124432.65,NaN,florin,1390-01-01,Loans Receivable - Government,124432.65,Loan to Venice for Lombardy Wars operations,2,war_financing
2,Bruges,Gold Merchant,Cash,NaN,42534.83,NaN,florin,1390-01-01,Loans Receivable,42534.83,Loan issued to Gold Merchant from Bruges branch,3,loan_issuance
3,Florence,Republic of Florence,Cash,NaN,1617678.46,NaN,florin,1390-01-01,Loans Receivable - Government,1617678.46,War financing for Florentine operations agains...,4,war_financing
4,Florence,Republic of Florence,Cash,NaN,27742.84,NaN,florin,1390-01-01,Loans Receivable - Government,27742.84,Loan to Venice for Lombardy Wars operations,5,war_financing


In [25]:
df_json.tail()

,branch,counterparty,credit_account,credit_account_2,credit_amount,credit_amount_2,currency,date,debit_account,debit_amount,description,id,type
80225,Geneva,Transfer to Venice,Cash,Exchange Fee Revenue,3091.96,71.82,florin,1440-12-30,Due from Venice,3163.78,Bill of exchange from Geneva to Venice,80226,bill_of_exchange
80226,Florence,Republic of Florence,Cash,NaN,197709.59,NaN,florin,1440-12-30,Loans Receivable - Government,197709.59,Loan to Venice for Lombardy Wars operations,80227,war_financing
80227,Venice,Marquis,Loans Receivable,Interest Income,120700.73,28968.18,florin,1440-12-31,Cash,149668.91,Loan repayment from Marquis with interest,80228,loan_repayment
80228,Geneva,Gold Merchant,Cash,,69318.55,,florin,1440-12-31,Loans Receivable,69318.55,Loan issued to Gold Merchant from Geneva branch,80229,loan_issuance
80229,Milan,Wine Trader,Loans Receivable,Interest Income,313446.76,72092.75,florin,1440-12-31,Cash,385539.51,Loan repayment from Wine Trader with interest,80230,loan_repayment


In [26]:
df_json.shape

(80230, 13)

In [27]:
df_json.info()

<class 'pandas.DataFrame'>
RangeIndex: 80230 entries, 0 to 80229
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   branch            80230 non-null  str           
 1   counterparty      80230 non-null  str           
 2   credit_account    80230 non-null  str           
 3   credit_account_2  30930 non-null  str           
 4   credit_amount     80230 non-null  float64       
 5   credit_amount_2   30930 non-null  object        
 6   currency          80230 non-null  str           
 7   date              80230 non-null  datetime64[us]
 8   debit_account     80230 non-null  str           
 9   debit_amount      80230 non-null  float64       
 10  description       80230 non-null  str           
 11  id                80230 non-null  int64         
 12  type              80230 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(1), str(8)
memory usage: 8.0+ MB


In [28]:
df_json.isnull().sum()

branch                  0
counterparty            0
credit_account          0
credit_account_2    49300
credit_amount           0
credit_amount_2     49300
currency                0
date                    0
debit_account           0
debit_amount            0
description             0
id                      0
type                    0
dtype: int64

In [30]:
df_json["date"] = pd.to_datetime(df_json["date"])

print("Date data type:", df_json["date"].dtype)
print("Earliest transaction:", df_json["date"].min())
print("Latest transaction:", df_json["date"].max())

Date data type: datetime64[us]
Earliest transaction: 1390-01-01 00:00:00
Latest transaction: 1440-12-31 00:00:00


In [31]:
df_json["credit_account_2"].isnull().sum()

np.int64(49300)

6. **CSV vs JSON Cross Validation:**
- Purpose: to confirm the data in the CSV and JSON files have the same underlying data.
- Why: Confirms the 2 export files can be used in later phases of the project if one needs to be favored over the other; so both can be trusted
- Method: Compared len() to count the rows, and used set()/value counts() on `id` and `type` to confirm matching transactions and categories
- Findings: four checks confirmed True: row count match (80,230 = 80,230), transaction ID sets match exactly, `type` category sets match, and `type` value_counts match per-category. Confirms CSV and JSON represent the same 80,230 transactions. One structural difference noted despite the match: `credit_account_2`/`credit_amount_2` null counts differ between files (CSV: 65,632 null; JSON: 49,300 null) — the JSON encodes some blank values as empty strings rather than true nulls, inconsistently within the same file (see Section 5). This doesn't affect transaction identity, but ingestion code will need to handle both blank representations for this field.

In [32]:
len(df_csv) == len(df_json) #shows if both files have the same row count

True

In [33]:
set(df_csv["id"]) == set(df_json["id"]) #shows if both files have the same id's or transactions

True

In [37]:
set(df_csv["type"]) == set(df_json["type"]) # shows is both files have the same categories?



True

In [38]:
df_csv["type"].value_counts().sort_index().equals(df_json["type"].value_counts().sort_index())  # shows if both files have same counts per category?


True